# Mortgage Prepayment Simulation — Monte Carlo and Stochastic Rate Modelling
### Mortgage Engine

This is the third notebook of the project **"Mortgage Prepayment Simulation — Monte Carlo and Stochastic Rate Modelling"**

This notebook defines functions to load loan-level information and model the characteristic of each loan. We model the EMI, the age of the loan, the Outstanding UPB as at any date, and prepayment incentive depending on the spread. 

The notebook creates helper functions to estimate attributes of each loan which is then exported to the final simulation engine to model prepayment rates. 

In [ ]:
#Import necessary libraries
import pandas as pd
import numpy as np

In [ ]:
#Load loan-level mortgage information and characteristics
mortgage_data = pd.read_excel("00_MortgageData.xlsx")

In [ ]:
#Define a function to estimate the monthly instalment given a loan's original balance, annual rate of interest and term of the loan
def monthly_payment(loan_amount, annual_rate, term):
    rate = annual_rate/12
    return loan_amount*rate/(1-((1+rate)**(-term)))

In [ ]:
#Define a function to find the age of the loan (in months) given its inception date and current evaluation date 
def age_of_the_loan(eval_date, inception_date):
    return (eval_date.month-inception_date.month)+(eval_date.year-inception_date.year)*12

In [ ]:
#Define a function to estimate the UPB of a loan as at any date 
def loan_upb_at_given_date(age, orig_loan_bal, annual_rate, total_payments, prepayment_flag):
    if prepayment_flag==1 or age>total_payments:
        return 0
    else:
        loan_upb = orig_loan_bal*(((1+annual_rate/12)**total_payments)-((1+annual_rate/12)**age))/(((1+annual_rate/12)**total_payments)-1)
        return loan_upb

In [ ]:
#Define a function to estimate the total scheduled principal payments between two ages of a loan
def scheduled_principals(start_age, end_age,orig_loan_bal, annual_rate, total_payments):
    loan_upb_start_age = orig_loan_bal*(((1+annual_rate/12)**total_payments)-((1+annual_rate/12)**start_age))/(((1+annual_rate/12)**total_payments)-1)
    loan_upb_end_age = orig_loan_bal*(((1+annual_rate/12)**total_payments)-((1+annual_rate/12)**end_age))/(((1+annual_rate/12)**total_payments)-1)
    return loan_upb_start_age-loan_upb_end_age

In [ ]:
#Model Prepyament economics and incentive. 
#Simple prepayment probability is estimated only on the basis of a base incentive, and FICO/LTV multiplier. 
def fico_multiplier_for_prepayments(fico):
    if fico>750:
        return 1.1
    elif 700<=fico<=750:
        return 0.75
    else:
        return 0.25

def ltv_multiplier_for_prepayments(ltv):
    if ltv>=95:
        return 0.25
    elif 85<=ltv<95:
        return 0.75
    else:
        return 1.1 

def prepay_flag(rate_spread, ltv, fico):
    scalar = 20
    base_prob = 1/(1+np.exp(-scalar*rate_spread))
    fico_multiplier = fico_multiplier_for_prepayments(fico)
    ltv_multiplier = ltv_multiplier_for_prepayments(ltv)
    final_prob = min(1,base_prob*fico_multiplier*ltv_multiplier)
    if rate_spread<0:
        return 0
    elif rate_spread>=0 and final_prob > np.random.uniform():
        return 1
    else:
        return 0

### End of Notebook